<a href="https://colab.research.google.com/github/e23281-lgtm/Statistical-Learning-e23281/blob/main/Data_wrangling_assignment_E23281.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Building a Modular Data Sanitization & Exploration Engine

**Reg. No:** E23281  
**Course:** Introduction to Statistical Learning  
**Dataset Used:** Titanic (Kaggle)

---

## Overview

A reusable Python toolkit for automated data cleaning, exploration, and
interactive visualization inside Google Colab. Accepts any CSV file and
provides a full pipeline from raw ingestion to ML-ready feature matrices.

## Features

- **Intelligent Loading**: Handles garbage strings (`?`, `NULL`, `n/a`) and
  auto-corrects column types
- **Structural Analysis**: Missing value reports, duplicate detection,
  statistical summaries
- **Automated Cleaning**: Mean/median/mode/constant imputation, IQR outlier
  detection, interactive row/column deletion
- **Feature Engineering**: MinMax, Standard, Robust scaling;
  OneHot, Ordinal, Uniform encoding
- **Interactive Visualizations**: Plotly-powered violin plots, scatter plots,
  histograms, and relationship charts
- **Deep Statistical Insights**: Unified heatmap combining Pearson r,
  Cramér's V, and Eta² across mixed data types
- **Custom Chart Generation**: `PlottingMethods` class returns embeddable HTML

## Project Structure

```
notebook/
├── Cell 1   : This overview (README)
├── Cell 2-3 : Installation and imports
├── Cell 4   : DataInspector class
├── Cell 5   : PlottingMethods class
└── Cell 6+  : End-to-end demonstration on Titanic dataset
```

In [1]:
# Install required library not included in Colab by default
!pip install statsmodels --quiet

In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import urllib.request
from plotly.subplots import make_subplots
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from scipy.stats import chi2_contingency, f_oneway, pearsonr
from google.colab import files
import io

print(" All libraries imported successfully.")

 All libraries imported successfully.


## Class 1: DataInspector

The core engine. Handles data ingestion, sanitization, structural analysis,
cleaning, feature engineering, and visualization. All state is stored in
`self.df` — load once, use all methods on the same object.

In [3]:
class DataInspector:
    """
    Core data pipeline engine for CSV ingestion, cleaning, feature engineering,
    and visualization. All state is stored in ``self.df`` — load once, apply all
    methods on the same instance.

    Attributes
    ----------
    df : pd.DataFrame or None
        The active working dataset. None until data is loaded.
    numeric_df : pd.DataFrame or None
        Scaled numeric columns produced by extract_normalized_numeric_data().
    categorical_df : pd.DataFrame or None
        Encoded categorical columns produced by extract_normalized_categorical_data().
    garbage_values : list of str
        Strings treated as NaN during CSV ingestion.
    """

    def __init__(self):
        self.df = None
        self.numeric_df = None
        self.categorical_df = None
        self.garbage_values = [
            '?', 'n/a', 'NA', 'N/A', 'NULL', 'null',
            'none', 'None', 'nan', 'NaN', '', ' ',
            'missing', 'MISSING', '-', '--'
        ]

    # ──────────────────────────────────────────────────────────
    # SECTION 1 · Data Ingestion
    # ──────────────────────────────────────────────────────────

    def upload_data(self):
        """
        Prompt a file-upload dialog (Colab only) and load the selected CSV.

        After loading, runs garbage-string cleaning and automatic type correction.
        Prints a shape summary and a 5-row preview on success.
        """
        print("Please upload your CSV file...")
        uploaded = files.upload()

        if not uploaded:
            print("No file was uploaded.")
            return

        filename = list(uploaded.keys())[0]
        self.df = pd.read_csv(
            io.BytesIO(uploaded[filename]),
            na_values=self.garbage_values,
            keep_default_na=True,
            na_filter=True
        )
        print(f" '{filename}' loaded — {self.df.shape[0]} rows × {self.df.shape[1]} columns")
        self._clean_garbage_strings()
        self._auto_correct_types()
        print("\n Preview (first 5 rows):")
        print(self.df.head())

    def load_from_url(self, url):
        """
        Load a publicly accessible CSV directly from a URL.

        Preferred for shared or automated notebooks where a manual upload
        is impractical. Applies the same post-load cleaning as upload_data().

        Parameters
        ----------
        url : str
            Direct link to a raw CSV file.
            Example::

                inspector.load_from_url(
                    'https://raw.githubusercontent.com/.../titanic.csv'
                )
        """
        if not url or not isinstance(url, str):
            print("Please provide a valid URL string.")
            return
        try:
            self.df = pd.read_csv(
                url,
                na_values=self.garbage_values,
                keep_default_na=True,
                na_filter=True
            )
            print(f" Data loaded — {self.df.shape[0]} rows × {self.df.shape[1]} columns")
            self._clean_garbage_strings()
            self._auto_correct_types()
            print("\n Preview (first 5 rows):")
            print(self.df.head())
        except Exception as e:
            print(f" Failed to load from URL: {e}")

    def drop_columns(self, columns):
        """
        Drop a list of columns without interactive input.

        Use this method in scripts. For interactive sessions where you want to
        type column names at runtime, use delete_columns() instead.

        Parameters
        ----------
        columns : list of str
            Column names to remove. Unrecognised names are skipped with a warning.

        Example::

            inspector.drop_columns(['Ticket', 'PassengerId', 'Name'])
        """
        if self.df is None:
            print("No data loaded.")
            return

        valid   = [c for c in columns if c in self.df.columns]
        invalid = [c for c in columns if c not in self.df.columns]

        if invalid:
            print(f"  Not found (skipped): {invalid}")
        if not valid:
            print("No valid columns to drop.")
            return

        self.df.drop(columns=valid, inplace=True)
        print(f"  Dropped: {valid}")
        print(f"   Remaining: {self.df.columns.tolist()}")

    def drop_rows(self, indices):
        """
        Drop a list of row indices without interactive input.

        Use this method in scripts. For interactive sessions, use delete_rows().
        Index is reset to 0-based integers after deletion.

        Parameters
        ----------
        indices : list of int
            Row indices to remove. Out-of-range values are skipped with a warning.

        Example::

            inspector.drop_rows([0, 5, 23])
        """
        if self.df is None:
            print("No data loaded.")
            return

        valid   = [i for i in indices if i in self.df.index]
        invalid = [i for i in indices if i not in self.df.index]

        if invalid:
            print(f"  Indices not found (skipped): {invalid}")
        if not valid:
            print("No valid indices to drop.")
            return

        self.df.drop(index=valid, inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        print(f"  Dropped {len(valid)} rows. Dataset now has {len(self.df)} rows.")

    def _clean_garbage_strings(self):
        """
        Strip surrounding whitespace and replace known garbage tokens with NaN.

        Operates only on object (string) columns. Called automatically after
        every load method so callers never need to invoke it directly.
        """
        text_cols = self.df.select_dtypes(include=['object']).columns
        garbage_map = {g: np.nan for g in self.garbage_values}
        for col in text_cols:
            self.df[col] = self.df[col].str.strip().replace(garbage_map)
        print(" Garbage-string cleaning complete.")

    def _auto_correct_types(self):
        """
        Coerce columns to numeric types where the conversion is non-trivial.

        A column is only coerced when at least one value survives pd.to_numeric()
        without becoming NaN — this prevents silently destroying columns that
        happen to start with a numeric token (e.g., ticket numbers like '113803').
        """
        for col in self.df.columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            # Only replace if the conversion preserved meaningful data.
            if not converted.isna().all():
                self.df[col] = converted
        print(" Auto-type correction complete.")
        print("\nColumn dtypes after correction:")
        print(self.df.dtypes)

    # ──────────────────────────────────────────────────────────
    # SECTION 2 · Structural Analysis
    # ──────────────────────────────────────────────────────────

    def summary(self):
        """
        Print a full structural audit of the loaded dataset.

        Reports: shape, per-column missing-value counts and percentages,
        numeric and categorical column lists, duplicate row count,
        descriptive statistics, and a 20-row preview.
        """
        if self.df is None:
            print("No data loaded. Please call upload_data() or load_from_url() first.")
            return

        rows, cols = self.df.shape
        missing_counts = self.df.isnull().sum()
        missing_cols   = missing_counts[missing_counts > 0]

        print("=" * 60)
        print("  DATASET SUMMARY")
        print("=" * 60)
        print(f"\n Shape: {rows} rows × {cols} columns")
        print(f"\n Missing Values: {missing_counts.sum()} total across {len(missing_cols)} columns")

        if len(missing_cols):
            print("\n   Column              | Missing Count | % of Rows")
            print("   " + "-" * 47)
            for col_name, count in missing_cols.items():
                print(f"   {col_name:<20} | {count:>5}         | {count / rows * 100:.1f}%")
        else:
            print("No missing values found!")

        numeric_cols     = self.df.select_dtypes(include=[np.number]).columns.tolist()
        categorical_cols = self.df.select_dtypes(include=['object']).columns.tolist()

        print(f"\n Numeric ({len(numeric_cols)}):     {numeric_cols}")
        print(f" Categorical ({len(categorical_cols)}): {categorical_cols}")
        print(f"\n Duplicate rows: {self.df.duplicated().sum()}")
        print("\n Numeric Statistics:")
        print(self.df.describe().round(2))
        print(f"\n First 20 rows:")
        print(self.df.head(20))
        print("\n" + "=" * 60)

    def handle_missing_values(self, strategy='mean', columns=None, constant=None):
        """
        Impute missing values using the chosen strategy.

        Parameters
        ----------
        strategy : {'mean', 'median', 'mode', 'constant'}
            Imputation method. ``mean`` and ``median`` are restricted to
            numeric columns when ``columns`` is not specified explicitly.
        columns : list of str or None
            Columns to impute. When None, all applicable columns are selected
            automatically based on the strategy.
        constant : scalar or None
            Required when ``strategy='constant'``. The fill value.

        Notes
        -----
        ``median`` is preferred over ``mean`` for right-skewed distributions
        (e.g., Fare, income) because a small number of extreme values can pull
        the mean far from the typical observation.
        """
        if self.df is None:
            print("No data loaded.")
            return

        if columns is None:
            if strategy in ['mean', 'median']:
                columns = self.df.select_dtypes(include=[np.number]).columns.tolist()
            else:
                columns = self.df.columns[self.df.isnull().any()].tolist()

        for col in columns:
            before = self.df[col].isnull().sum()
            if before == 0:
                continue

            if strategy == 'mean':
                fill_value = self.df[col].mean()
            elif strategy == 'median':
                fill_value = self.df[col].median()
            elif strategy == 'mode':
                mode_result = self.df[col].mode()
                if len(mode_result) == 0:
                    print(f"  '{col}' is entirely NaN — skipping.")
                    continue
                fill_value = mode_result[0]
            elif strategy == 'constant':
                if constant is None:
                    print("Error: strategy='constant' requires constant=<value>.")
                    return
                fill_value = constant
            else:
                print(f"Unknown strategy '{strategy}'. Choose: mean, median, mode, constant.")
                return

            self.df[col] = self.df[col].fillna(fill_value)
            display = round(fill_value, 4) if isinstance(fill_value, float) else fill_value
            print(f" '{col}': filled {before} NaNs with {strategy} = {display}")

        print(f"\n✔ Imputation complete (strategy='{strategy}').")

    def remove_duplicates(self):
        """
        Remove exact duplicate rows, keeping the first occurrence.

        The index is reset to a clean 0-based integer sequence after removal.
        """
        if self.df is None:
            print("No data loaded.")
            return

        before = len(self.df)
        self.df.drop_duplicates(keep='first', inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        removed = before - len(self.df)

        if removed == 0:
            print(" No duplicate rows found.")
        else:
            print(f"  Removed {removed} duplicate rows. Dataset now has {len(self.df)} rows.")

    def handle_outliers(self, columns, action='flag'):
        """
        Detect outliers via the IQR fence method (Tukey, 1977).

        A value is an outlier if it falls below Q1 − 1.5·IQR or above
        Q3 + 1.5·IQR. Rows outlying in multiple columns are counted once.

        Parameters
        ----------
        columns : list of str
            Numeric columns to inspect.
        action : {'flag', 'remove'}
            ``flag``   — adds a boolean ``is_outlier`` column (non-destructive).
                         Inspect with ``inspector.df[inspector.df['is_outlier']]``.
            ``remove`` — permanently deletes outlier rows and resets the index.
        """
        if self.df is None:
            print("No data loaded.")
            return

        # Use a set so a row outlying in multiple columns is counted only once.
        outlier_indices = set()

        for col in columns:
            if not pd.api.types.is_numeric_dtype(self.df[col]):
                print(f"  '{col}' is not numeric — skipping.")
                continue

            Q1, Q3 = self.df[col].quantile(0.25), self.df[col].quantile(0.75)
            IQR    = Q3 - Q1
            lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
            mask = (self.df[col] < lower) | (self.df[col] > upper)

            col_outliers = self.df.index[mask].tolist()
            print(f" '{col}': {len(col_outliers)} outliers (outside [{lower:.2f}, {upper:.2f}])")
            outlier_indices.update(col_outliers)

        total = len(outlier_indices)
        print(f"\n   Total unique outlier rows: {total}")

        if total == 0:
            print(" No outliers detected.")
            return

        if action == 'flag':
            self.df['is_outlier'] = self.df.index.isin(outlier_indices)
            print(" Flagged in new column 'is_outlier'.")
        elif action == 'remove':
            before = len(self.df)
            self.df.drop(list(outlier_indices), axis=0, inplace=True)
            self.df.reset_index(drop=True, inplace=True)
            print(f"  Removed {before - len(self.df)} rows. Dataset now has {len(self.df)} rows.")
        else:
            print(f"Unknown action '{action}'. Use 'flag' or 'remove'.")

    def delete_rows(self):
        """
        Interactively prompt the user for row indices to delete.

        Intended for exploratory sessions. For scripted pipelines use
        drop_rows() instead. Accepts a comma-separated list of integers.

        Example prompt input::

            0, 5, 23, 100
        """
        if self.df is None:
            print("No data loaded.")
            return

        print(f"Dataset has {len(self.df)} rows (indices 0–{len(self.df) - 1}).")
        user_input = input("Enter row indices to delete (comma-separated): ")

        if not user_input.strip():
            print("Nothing entered. No rows deleted.")
            return

        try:
            indices = [int(i.strip()) for i in user_input.split(',')]
        except ValueError:
            print("Invalid input — please enter integers only, separated by commas.")
            return

        valid   = [i for i in indices if i in self.df.index]
        invalid = [i for i in indices if i not in self.df.index]

        if invalid:
            print(f"  Indices not found (skipped): {invalid}")
        if not valid:
            print("No valid indices to delete.")
            return

        self.df.drop(valid, axis=0, inplace=True)
        self.df.reset_index(drop=True, inplace=True)
        print(f"  Deleted {len(valid)} rows. Dataset now has {len(self.df)} rows.")

    def delete_columns(self):
        """
        Interactively prompt the user for column names to delete.

        Intended for exploratory sessions. For scripted pipelines use
        drop_columns() instead. Accepts a comma-separated list of names.

        Example prompt input::

            Cabin, Ticket, PassengerId
        """
        if self.df is None:
            print("No data loaded.")
            return

        print(f"Available columns: {self.df.columns.tolist()}")
        user_input = input("Enter column names to delete (comma-separated): ")

        if not user_input.strip():
            print("Nothing entered. No columns deleted.")
            return

        cols_to_delete = [c.strip() for c in user_input.split(',')]
        valid   = [c for c in cols_to_delete if c in self.df.columns]
        invalid = [c for c in cols_to_delete if c not in self.df.columns]

        if invalid:
            print(f"  Not found (skipped): {invalid}")
        if not valid:
            print("No valid columns to delete.")
            return

        self.df.drop(valid, axis=1, inplace=True)
        print(f"  Deleted: {valid}")
        print(f"   Remaining: {self.df.columns.tolist()}")

    # ──────────────────────────────────────────────────────────
    # SECTION 3 · Feature Engineering
    # ──────────────────────────────────────────────────────────

    def extract_normalized_numeric_data(self, method='minmax'):
        """
        Scale all numeric columns and store the result in ``self.numeric_df``.

        The ``Survived`` column is excluded from scaling if present — scaling
        a binary target label would corrupt the class values fed into a classifier.

        Parameters
        ----------
        method : {'minmax', 'standard', 'robust'}
            ``minmax``   — scales each column to [0, 1].
            ``standard`` — Z-score normalisation (mean=0, std=1).
                           Best default for most ML algorithms.
            ``robust``   — median/IQR-based scaling; resists outlier influence.

        Returns
        -------
        pd.DataFrame
            Scaled numeric columns (also stored in ``self.numeric_df``).
        """
        if self.df is None:
            print("No data loaded.")
            return None

        numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        if 'Survived' in numeric_cols:
            numeric_cols.remove('Survived')

        if not numeric_cols:
            print("No numeric columns found to scale.")
            return None

        scalers = {'minmax': MinMaxScaler, 'standard': StandardScaler, 'robust': RobustScaler}
        if method not in scalers:
            print(f"Unknown method '{method}'. Choose: minmax, standard, robust.")
            return None

        scaled = scalers[method]().fit_transform(self.df[numeric_cols].copy())
        self.numeric_df = pd.DataFrame(scaled, columns=numeric_cols, index=self.df.index)

        print(f" Scaled using '{method}'. Columns: {numeric_cols}")
        print(f"\n   Preview (first 3 rows):")
        print(self.numeric_df.head(3).round(4))
        return self.numeric_df

    def extract_normalized_categorical_data(self, method='onehot'):
        """
        Encode all categorical (object) columns and store the result in
        ``self.categorical_df``.

        Parameters
        ----------
        method : {'onehot', 'ordinal', 'uniform'}
            ``onehot``  — one binary column per unique value (correct for
                          unordered categories; avoids implying numeric rank).
            ``ordinal`` — integer codes 0, 1, 2 …  (use only when a natural
                          ordering exists, e.g., low/medium/high).
            ``uniform`` — frequency-rank mapped to 0.0–1.0 (most-frequent
                          category → 0.0, least-frequent → 1.0).

        Returns
        -------
        pd.DataFrame
            Encoded columns (also stored in ``self.categorical_df``).
        """
        if self.df is None:
            print("No data loaded.")
            return None

        cat_cols = self.df.select_dtypes(include=['object']).columns.tolist()
        if not cat_cols:
            print("No categorical columns found to encode.")
            return None

        if method == 'onehot':
            enc = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
            arr = enc.fit_transform(self.df[cat_cols])
            self.categorical_df = pd.DataFrame(
                arr, columns=enc.get_feature_names_out(cat_cols), index=self.df.index
            )

        elif method == 'ordinal':
            arr = OrdinalEncoder().fit_transform(self.df[cat_cols])
            self.categorical_df = pd.DataFrame(arr, columns=cat_cols, index=self.df.index)

        elif method == 'uniform':
            frames = []
            for col in cat_cols:
                counts = self.df[col].value_counts()
                mapping = {cat: v for v, cat in zip(np.linspace(0, 1, len(counts)), counts.index)}
                frames.append(self.df[col].map(mapping))
            self.categorical_df = pd.concat(frames, axis=1)

        else:
            print(f"Unknown method '{method}'. Choose: onehot, ordinal, uniform.")
            return None

        print(f" Encoded using '{method}'. Shape: {self.categorical_df.shape}")
        print(f"\n   Preview (first 3 rows):")
        print(self.categorical_df.head(3).round(4))
        return self.categorical_df

    def merge_normalized_data(self):
        """
        Concatenate scaled numeric and encoded categorical DataFrames into a
        single ML-ready feature matrix.

        The ``Survived`` target column is re-attached from the original
        unscaled ``self.df`` so its 0/1 labels are preserved exactly.

        Returns
        -------
        pd.DataFrame
            Combined feature matrix ready for ML algorithms.
        """
        if self.numeric_df is None or self.categorical_df is None:
            print("Run extract_normalized_numeric_data() and "
                  "extract_normalized_categorical_data() before merging.")
            return None

        merged = pd.concat([self.numeric_df, self.categorical_df], axis=1)
        if 'Survived' in self.df.columns:
            merged['Survived'] = self.df['Survived'].values

        print(f" Merged shape: {merged.shape}")
        print(f"   Columns: {merged.columns.tolist()}")
        print(f"\n   Preview (first 3 rows):")
        print(merged.head(3).round(4))
        return merged

    # ──────────────────────────────────────────────────────────
    # SECTION 4 · Visualization
    # ──────────────────────────────────────────────────────────

    def plot_univariate(self, column):
        """
        Render a 3-panel subplot for a single numeric column:

        * **Panel 1** — Horizontal violin + embedded box plot.
        * **Panel 2** — Scatter of row-index vs value (reveals ordering effects).
        * **Panel 3** — Histogram with 30 bins.

        Key statistics (mean, median, std, missing count) are embedded in
        the figure title so no separate print statement is needed.

        Parameters
        ----------
        column : str
            Name of a numeric column in ``self.df``.
        """
        if self.df is None:
            print("No data loaded.")
            return
        if column not in self.df.columns:
            print(f"'{column}' not found. Available: {self.df.columns.tolist()}")
            return
        if not pd.api.types.is_numeric_dtype(self.df[column]):
            print(f"'{column}' is not numeric. Use plot_categorical_frequency() instead.")
            return

        d = self.df[column].dropna()

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=('Distribution Shape', 'Value per Row', 'Frequency'),
            horizontal_spacing=0.08
        )

        fig.add_trace(go.Violin(
            x=d, name=column, box_visible=True, meanline_visible=True,
            fillcolor='rgba(99,110,250,0.4)', line_color='rgb(99,110,250)',
            hovertemplate=f'{column}: %{{x:.2f}}<extra></extra>'
        ), row=1, col=1)

        fig.add_trace(go.Scatter(
            x=d.index, y=d.values, mode='markers',
            marker=dict(color=d.values, colorscale='Viridis', size=4,
                        opacity=0.6, showscale=True, colorbar=dict(thickness=8)),
            hovertemplate=f'Row: %{{x}}<br>{column}: %{{y:.2f}}<extra></extra>'
        ), row=1, col=2)

        fig.add_trace(go.Histogram(
            x=d, nbinsx=30, marker_color='rgba(99,110,250,0.7)',
            marker_line=dict(color='white', width=0.5),
            hovertemplate='Range: %{x}<br>Count: %{y}<extra></extra>'
        ), row=1, col=3)

        fig.update_layout(
            title=dict(
                text=(f"<b>Univariate Analysis: {column}</b><br>"
                      f"<sup>mean={d.mean():.2f} | median={d.median():.2f} | "
                      f"std={d.std():.2f} | missing={self.df[column].isnull().sum()}</sup>"),
                x=0.5, font_size=16
            ),
            height=420, showlegend=False, template='plotly_white'
        )
        fig.update_xaxes(title_text="Row Index", row=1, col=2)
        fig.update_yaxes(title_text=column,      row=1, col=2)
        fig.update_xaxes(title_text=column,      row=1, col=3)
        fig.update_yaxes(title_text="Count",     row=1, col=3)
        fig.show()

    def plot_relationship(self, col1, col2):
        """
        Plot the relationship between two columns, auto-selecting chart type.

        Type-detection logic
        --------------------
        A numeric column with ≤ 10 unique values is reclassified as categorical.
        This prevents misleadingly treating ``Pclass`` (1, 2, 3) or ``Survived``
        (0, 1) as continuous axes.

        Chart selection
        ---------------
        =================== ======================
        Column types        Chart produced
        =================== ======================
        Numeric × Numeric   Scatter + OLS trendline + Pearson r
        Cat × Numeric       Box plot with jittered points
        Numeric × Cat       Box plot (axes swapped)
        Cat × Cat           Grouped bar (cross-tabulation)
        =================== ======================

        Parameters
        ----------
        col1 : str
        col2 : str
        """
        if self.df is None:
            print("No data loaded.")
            return
        for col in [col1, col2]:
            if col not in self.df.columns:
                print(f"'{col}' not found.")
                return

        def is_numeric(col):
            return (pd.api.types.is_numeric_dtype(self.df[col])
                    and self.df[col].nunique() > 10)

        c1_num = is_numeric(col1)
        c2_num = is_numeric(col2)
        plot_df = self.df[[col1, col2]].dropna()

        if c1_num and c2_num:
            fig = px.scatter(plot_df, x=col1, y=col2, trendline='ols',
                             opacity=0.6, color_discrete_sequence=['rgb(99,110,250)'],
                             template='plotly_white')
            r, p = pearsonr(plot_df[col1], plot_df[col2])
            fig.update_layout(
                title=f'<b>{col1} vs {col2}</b>  <sup>Pearson r={r:.3f} | p={p:.4f}</sup>',
                height=500
            )

        elif not c1_num and c2_num:
            fig = px.box(plot_df, x=col1, y=col2, points='all', color=col1,
                         title=f'<b>{col2} by {col1}</b>', template='plotly_white')
            fig.update_layout(height=500, showlegend=False)

        elif c1_num and not c2_num:
            fig = px.box(plot_df, x=col2, y=col1, points='all', color=col2,
                         title=f'<b>{col1} by {col2}</b>', template='plotly_white')
            fig.update_layout(height=500, showlegend=False)

        else:
            ct_long = (pd.crosstab(plot_df[col1], plot_df[col2])
                       .reset_index()
                       .melt(id_vars=col1, var_name=col2, value_name='Count'))
            fig = px.bar(ct_long, x=col1, y='Count', color=col2, barmode='group',
                         title=f'<b>{col1} vs {col2}</b>', template='plotly_white')
            fig.update_layout(height=500)

        fig.show()

    def plot_categorical_frequency(self, column):
        """
        Render a bar chart with count and percentage labels for a categorical column.

        The chart title includes total row count and the number of missing values
        in the column so data quality is visible at a glance.

        Parameters
        ----------
        column : str
            Column name (typically object dtype).
        """
        if self.df is None:
            print("No data loaded.")
            return
        if column not in self.df.columns:
            print(f"'{column}' not found.")
            return

        counts = self.df[column].value_counts()
        total  = len(self.df)
        pct    = (counts / total * 100).round(1)

        freq_df = pd.DataFrame({
            'Category':   counts.index.astype(str),
            'Count':      counts.values,
            'Percentage': pct.values
        })

        fig = px.bar(
            freq_df, x='Category', y='Count', text='Percentage',
            color='Count', color_continuous_scale='Blues',
            title=(f'<b>Frequency: {column}</b>  '
                   f'<sup>({total} rows, {self.df[column].isnull().sum()} missing)</sup>'),
            template='plotly_white',
            category_orders={'Category': freq_df['Category'].tolist()}
        )
        fig.update_traces(texttemplate='%{text}%', textposition='outside')
        fig.update_layout(height=450, coloraxis_showscale=False,
                          yaxis_title='Count', xaxis_title=column)
        fig.show()

    # ──────────────────────────────────────────────────────────
    # SECTION 5 · Statistical Insights
    # ──────────────────────────────────────────────────────────

    def plot_all_associations_heatmap(self):
        """
        Render a unified association heatmap across all column pairs.

        The heatmap mixes three statistical measures depending on column types:

        ======================= ========================================
        Column pair             Measure used
        ======================= ========================================
        Numeric × Numeric       Pearson's r (range −1 to +1)
        Categorical × Categorical  Cramér's V (range 0 to +1)
        Numeric × Categorical   Eta² / ANOVA (range 0 to +1)
        ======================= ========================================

        The matrix is symmetric — only the upper triangle is computed;
        the lower triangle mirrors it to halve computation time.
        Hover text on each cell shows the exact measure used and the score.
        """
        if self.df is None:
            print("No data loaded.")
            return

        cols = self.df.columns.tolist()
        n    = len(cols)
        assoc  = np.zeros((n, n))
        method = [[""] * n for _ in range(n)]

        def cramers_v(a, b):
            """Symmetric association between two categorical columns (0–1)."""
            ct   = pd.crosstab(a, b)
            chi2 = chi2_contingency(ct)[0]
            k    = min(ct.shape)
            if k <= 1:
                return 0.0
            return round(float(np.sqrt(chi2 / (ct.values.sum() * (k - 1)))), 4)

        def eta_squared(num, cat):
            """Proportion of numeric variance explained by categorical grouping (0–1)."""
            groups = [num[cat == c].dropna().values for c in cat.dropna().unique()]
            groups = [g for g in groups if len(g) > 0]
            if len(groups) < 2:
                return 0.0
            F, _ = f_oneway(*groups)
            if np.isnan(F):
                return 0.0
            all_v      = num.dropna().values
            grand_mean = all_v.mean()
            ss_between = sum(len(g) * (g.mean() - grand_mean) ** 2 for g in groups)
            ss_total   = sum((v - grand_mean) ** 2 for v in all_v)
            return 0.0 if ss_total == 0 else round(float(np.clip(ss_between / ss_total, 0, 1)), 4)

        print("Computing associations…")

        for i, ci in enumerate(cols):
            for j, cj in enumerate(cols):
                if i == j:
                    assoc[i][j] = 1.0; method[i][j] = "self"; continue
                # Mirror — matrix is symmetric; skip already-computed pair.
                if assoc[j][i] != 0:
                    assoc[i][j] = assoc[j][i]; method[i][j] = method[j][i]; continue

                i_num = pd.api.types.is_numeric_dtype(self.df[ci])
                j_num = pd.api.types.is_numeric_dtype(self.df[cj])

                try:
                    if i_num and j_num:
                        v = self.df[[ci, cj]].dropna()
                        score = 0.0 if len(v) < 3 else round(float(pearsonr(v[ci], v[cj])[0]), 4)
                        method[i][j] = "Pearson r"
                    elif not i_num and not j_num:
                        v = self.df[[ci, cj]].dropna()
                        score = cramers_v(v[ci], v[cj])
                        method[i][j] = "Cramér V"
                    else:
                        num_c = self.df[ci] if i_num else self.df[cj]
                        cat_c = self.df[cj] if i_num else self.df[ci]
                        score = eta_squared(num_c, cat_c)
                        method[i][j] = "Eta²"
                except Exception:
                    score = 0.0; method[i][j] = "error"

                assoc[i][j] = score

        assoc_df   = pd.DataFrame(assoc, index=cols, columns=cols)
        hover_text = [
            [f"{cols[i]} vs {cols[j]}<br>Score: {assoc[i][j]:.3f}<br>Method: {method[i][j]}"
             for j in range(n)] for i in range(n)
        ]

        fig = go.Figure(data=go.Heatmap(
            z=assoc_df.values, x=cols, y=cols,
            text=assoc_df.values.round(2), texttemplate="%{text}",
            hovertext=hover_text, hovertemplate="%{hovertext}<extra></extra>",
            colorscale='RdBu_r', zmin=-1, zmax=1,
            colorbar=dict(title="Strength", thickness=15)
        ))
        fig.update_layout(
            title=dict(
                text=('<b>Unified Association Heatmap</b><br>'
                      '<sup>Pearson r (Num–Num) | Cramér V (Cat–Cat) | Eta² (Num–Cat)</sup>'),
                x=0.5, font_size=15
            ),
            height=600, width=700, template='plotly_white',
            xaxis=dict(tickangle=45)
        )
        fig.show()
        print("\n Heatmap rendered. Hover over cells to see the measure used.")


## Class 2: PlottingMethods

A standalone chart generation utility. Methods accept plain DataFrames and
return interactive Plotly charts as HTML strings for flexible embedding.
Each method also calls `fig.show()` to render inline in the notebook.

In [4]:
class PlottingMethods:
    """
    Standalone chart-generation utility.

    Unlike ``DataInspector``, this class holds no state. Every method accepts
    a plain DataFrame, renders an interactive Plotly chart inline, and returns
    the figure as an HTML fragment string for embedding outside the notebook.

    Usage
    -----
    ::

        plotter = PlottingMethods()
        html = plotter.bar_chart(df, x_col='Embarked', y_col='Count',
                                 title='Passengers by Port')
        plotter.save_html(html, 'port_chart.html')
    """

    def bar_chart(self, data, x_col, y_col, title='Bar Chart',
                  color_col=None, orientation='v'):
        """
        Generate an interactive bar chart.

        Parameters
        ----------
        data : pd.DataFrame
        x_col : str
            Column for the x-axis (categorical labels).
        y_col : str
            Column for the y-axis (numeric values).
        title : str, optional
            Chart title. Default ``'Bar Chart'``.
        color_col : str or None, optional
            Column used for grouped colour encoding. Pass ``None`` for a
            single-colour chart.
        orientation : {'v', 'h'}, optional
            ``'v'`` for vertical bars (default); ``'h'`` for horizontal.

        Returns
        -------
        str
            Self-contained HTML fragment (Plotly JS loaded from CDN).

        Example
        -------
        ::

            html = plotter.bar_chart(
                survivors_by_class, x_col='Pclass', y_col='Survived',
                title='Survivors by Passenger Class'
            )
        """
        for col in [x_col, y_col]:
            if col not in data.columns:
                print(f"Column '{col}' not found in the provided DataFrame.")
                return ""

        fig = px.bar(
            data, x=x_col, y=y_col, color=color_col,
            orientation=orientation,
            title=f'<b>{title}</b>',
            template='plotly_white',
            text_auto=True,
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        fig.update_layout(height=450, bargap=0.2,
                          xaxis_title=x_col, yaxis_title=y_col)
        fig.update_traces(textposition='outside')

        html = fig.to_html(full_html=False, include_plotlyjs='cdn')
        fig.show()
        return html

    def pie_chart(self, data, names_col, values_col, title='Pie Chart', hole=0.3):
        """
        Generate an interactive pie or donut chart.

        Parameters
        ----------
        data : pd.DataFrame
        names_col : str
            Column containing slice labels.
        values_col : str
            Column containing numeric slice sizes.
        title : str, optional
            Chart title. Default ``'Pie Chart'``.
        hole : float, optional
            Fractional radius of the centre hole.
            ``0`` → solid pie, ``0.3`` → donut (default), ``0.7`` → thin ring.

        Returns
        -------
        str
            HTML fragment string.

        Example
        -------
        ::

            port_counts = df['Embarked'].value_counts().reset_index()
            port_counts.columns = ['Port', 'Count']
            html = plotter.pie_chart(port_counts, names_col='Port',
                                     values_col='Count',
                                     title='Passengers by Embarkation Port')
        """
        for col in [names_col, values_col]:
            if col not in data.columns:
                print(f"Column '{col}' not found.")
                return ""

        fig = px.pie(
            data, names=names_col, values=values_col, hole=hole,
            title=f'<b>{title}</b>',
            template='plotly_white',
            color_discrete_sequence=px.colors.qualitative.Set2
        )
        fig.update_traces(
            textposition='inside',
            textinfo='percent+label',
            hovertemplate='<b>%{label}</b><br>Count: %{value}<br>'
                          'Percentage: %{percent}<extra></extra>'
        )
        fig.update_layout(height=450)

        html = fig.to_html(full_html=False, include_plotlyjs='cdn')
        fig.show()
        return html

    def histogram(self, data, column, title='Histogram', nbins=30, color=None):
        """
        Generate an interactive histogram for a numeric column.

        A marginal box plot is added above the histogram to surface the
        median, quartiles, and outliers in the same figure without a
        separate call.

        When ``color`` is supplied, distributions are overlaid with
        ``opacity=0.75`` so both layers remain visible. Use ``barmode='group'``
        in a manual ``px.histogram`` call if side-by-side bars are preferred
        over overlapping ones.

        Parameters
        ----------
        data : pd.DataFrame
        column : str
            Numeric column to plot.
        title : str, optional
            Chart title. Default ``'Histogram'``.
        nbins : int, optional
            Number of histogram bins. Default ``30``.
        color : str or None, optional
            Column for overlapping colour groups.
            Example: ``color='Survived'`` overlays survival outcomes.

        Returns
        -------
        str
            HTML fragment string.

        Example
        -------
        ::

            html = plotter.histogram(df, column='Age',
                                     title='Age Distribution by Survival',
                                     nbins=25, color='Survived')
        """
        if column not in data.columns:
            print(f"Column '{column}' not found.")
            return ""
        if not pd.api.types.is_numeric_dtype(data[column]):
            print(f"Column '{column}' is not numeric.")
            return ""

        fig = px.histogram(
            data, x=column, color=color, nbins=nbins,
            title=f'<b>{title}</b>',
            template='plotly_white',
            opacity=0.75,
            barmode='overlay',
            color_discrete_sequence=px.colors.qualitative.Set2,
            marginal='box'
        )
        fig.update_layout(height=450, xaxis_title=column,
                          yaxis_title='Count', bargap=0.05)

        html = fig.to_html(full_html=False, include_plotlyjs='cdn')
        fig.show()
        return html

    def save_html(self, html_string, filename='chart.html'):
        """
        Wrap an HTML fragment in a minimal full document and write it to disk.

        The output opens in any browser without Colab or a Python runtime.
        In Colab, download via **Files panel → right-click → Download**.

        Parameters
        ----------
        html_string : str
            HTML returned by ``bar_chart()``, ``pie_chart()``, or
            ``histogram()``.
        filename : str, optional
            Output file path. Default ``'chart.html'``.

        Example
        -------
        ::

            html = plotter.bar_chart(df, 'Sex', 'Fare')
            plotter.save_html(html, 'fare_by_sex.html')
        """
        if not html_string:
            print("Nothing to save — HTML string is empty.")
            return

        full_html = (
            f'<!DOCTYPE html>\n<html>\n'
            f'<head><meta charset="utf-8"><title>{filename}</title></head>\n'
            f'<body>\n{html_string}\n</body>\n</html>'
        )

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(full_html)

        print(f" Chart saved to '{filename}'")
        print("   Download from Colab: Files panel → right-click → Download")


---
## Demonstration: Full Pipeline on the Titanic Dataset

**Flow:** Load → Inspect → Clean → Normalize → Visualize → Associations

The Titanic dataset contains passenger records from the 1912 disaster.
891 rows, 12 columns. Target variable: `Survived` (0 = died, 1 = survived).

### Step 1 — Data Loading
Loading directly from a public URL so this notebook runs fully automatically
without any manual file upload.

In [5]:
inspector = DataInspector()
inspector.load_from_url(
    "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
)

 Data loaded — 891 rows × 12 columns
 Garbage-string cleaning complete.
 Auto-type correction complete.

Column dtypes after correction:
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket         float64
Fare           float64
Cabin           object
Embarked        object
dtype: object

 Preview (first 5 rows):
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs.

### Step 2 — Initial Inspection
Before touching anything, we audit the data: shape, missing values,
column types, duplicates, and statistical summaries.

In [6]:
inspector.summary()

  DATASET SUMMARY

 Shape: 891 rows × 12 columns

 Missing Values: 1096 total across 4 columns

   Column              | Missing Count | % of Rows
   -----------------------------------------------
   Age                  |   177         | 19.9%
   Ticket               |   230         | 25.8%
   Cabin                |   687         | 77.1%
   Embarked             |     2         | 0.2%

 Numeric (8):     ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare']
 Categorical (4): ['Name', 'Sex', 'Cabin', 'Embarked']

 Duplicate rows: 0

 Numeric Statistics:
       PassengerId  Survived  Pclass     Age   SibSp   Parch      Ticket  \
count       891.00    891.00  891.00  714.00  891.00  891.00      661.00   
mean        446.00      0.38    2.31   29.70    0.52    0.38   260318.55   
std         257.35      0.49    0.84   14.53    1.10    0.81   471609.27   
min           1.00      0.00    1.00    0.42    0.00    0.00      693.00   
25%         223.50      0.00    2.

### Step 3 — Data Cleaning
Drop uninformative columns, impute missing values, remove duplicates,
and audit outliers in the Fare column.

In [7]:
# Drop columns with no predictive value
inspector.drop_columns(['Ticket', 'PassengerId', 'Name'])

# Impute missing values
inspector.handle_missing_values(strategy='median',   columns=['Age'])
inspector.handle_missing_values(strategy='mode',     columns=['Embarked'])
inspector.handle_missing_values(strategy='constant', columns=['Cabin'],
                                 constant='Unknown')

# Remove duplicate rows
inspector.remove_duplicates()

# Audit outliers in Fare — flag to inspect, then remove the flag
inspector.handle_outliers(columns=['Fare'], action='flag')
print(f"\nFare outliers detected: {inspector.df['is_outlier'].sum()} rows")
print("Decision: KEEP — extreme fares are genuine first-class ticket prices.")
inspector.df.drop(columns=['is_outlier'], inplace=True)

  Dropped: ['Ticket', 'PassengerId', 'Name']
   Remaining: ['Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked']
 'Age': filled 177 NaNs with median = 28.0

✔ Imputation complete (strategy='median').
 'Embarked': filled 2 NaNs with mode = S

✔ Imputation complete (strategy='mode').
 'Cabin': filled 687 NaNs with constant = Unknown

✔ Imputation complete (strategy='constant').
  Removed 110 duplicate rows. Dataset now has 781 rows.
 'Fare': 102 outliers (outside [-30.91, 72.98])

   Total unique outlier rows: 102
 Flagged in new column 'is_outlier'.

Fare outliers detected: 102 rows
Decision: KEEP — extreme fares are genuine first-class ticket prices.


### Step 4 — Post-Cleaning Verification
Confirm zero missing values remain and review the cleaned dataset shape.

In [8]:
inspector.summary()

  DATASET SUMMARY

 Shape: 781 rows × 9 columns

 Missing Values: 0 total across 0 columns
No missing values found!

 Numeric (6):     ['Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare']
 Categorical (3): ['Sex', 'Cabin', 'Embarked']

 Duplicate rows: 0

 Numeric Statistics:
       Survived  Pclass     Age   SibSp   Parch    Fare
count    781.00  781.00  781.00  781.00  781.00  781.00
mean       0.41    2.24   29.62    0.52    0.42   34.75
std        0.49    0.86   13.76    0.99    0.84   52.24
min        0.00    1.00    0.42    0.00    0.00    0.00
25%        0.00    1.00   22.00    0.00    0.00    8.05
50%        0.00    3.00   28.00    0.00    0.00   15.90
75%        1.00    3.00   36.00    1.00    1.00   34.02
max        1.00    3.00   80.00    8.00    6.00  512.33

 First 20 rows:
    Survived  Pclass     Sex   Age  SibSp  Parch     Fare    Cabin Embarked
0          0       3    male  22.0      1      0   7.2500  Unknown        S
1          1       1  female  38.0      1      

### Step 5 — Feature Engineering
Scale numeric columns and encode categorical columns to prepare a
machine-learning-ready feature matrix.

In [9]:
# Standard scaling: mean=0, std=1. Best default for most ML algorithms.
inspector.extract_normalized_numeric_data(method='standard')

# One-hot encoding: correct for unordered categories (Sex, Embarked, Cabin)
inspector.extract_normalized_categorical_data(method='onehot')

# Merge into a single ML-ready DataFrame
ml_ready = inspector.merge_normalized_data()
print(f"\n Final ML-ready shape: {ml_ready.shape[0]} rows × {ml_ready.shape[1]} columns")

 Scaled using 'standard'. Columns: ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare']

   Preview (first 3 rows):
   Pclass     Age   SibSp   Parch    Fare
0  0.8882 -0.5542  0.4813 -0.4983 -0.5268
1 -1.4514  0.6090  0.4813 -0.4983  0.6998
2  0.8882 -0.2634 -0.5319 -0.4983 -0.5139
 Encoded using 'onehot'. Shape: (781, 153)

   Preview (first 3 rows):
   Sex_female  Sex_male  Cabin_A10  Cabin_A14  Cabin_A16  Cabin_A19  \
0         0.0       1.0        0.0        0.0        0.0        0.0   
1         1.0       0.0        0.0        0.0        0.0        0.0   
2         1.0       0.0        0.0        0.0        0.0        0.0   

   Cabin_A20  Cabin_A23  Cabin_A24  Cabin_A26  ...  Cabin_F2  Cabin_F33  \
0        0.0        0.0        0.0        0.0  ...       0.0        0.0   
1        0.0        0.0        0.0        0.0  ...       0.0        0.0   
2        0.0        0.0        0.0        0.0  ...       0.0        0.0   

   Cabin_F38  Cabin_F4  Cabin_G6  Cabin_T  Cabin_Unknown  Embarked_C

### Step 6 — Univariate Analysis
Inspect the distribution of individual numeric columns using
3-panel violin/scatter/histogram plots.

In [10]:
inspector.plot_univariate('Age')
inspector.plot_univariate('Fare')

### Step 7 — Relationship Analysis
Explore pairwise relationships. The method auto-selects the correct chart
type based on column data types.

In [11]:
inspector.plot_relationship('Sex', 'Survived')      # Cat-Num → Box plot
inspector.plot_relationship('Pclass', 'Survived')   # Cat-Cat (few unique) → Bar
inspector.plot_relationship('Age', 'Fare')          # Num-Num → Scatter + OLS

### Step 8 — Categorical Frequency Analysis
Show the distribution of values within categorical columns with counts
and percentage labels.

In [12]:
inspector.plot_categorical_frequency('Sex')
inspector.plot_categorical_frequency('Embarked')
inspector.plot_categorical_frequency('Pclass')

### Step 9 — Unified Association Heatmap
A single heatmap showing association strength between every pair of columns.
Uses three different statistical measures depending on column types:
- **Pearson r** for Numeric–Numeric pairs
- **Cramér's V** for Categorical–Categorical pairs  
- **Eta²** (ANOVA) for Numeric–Categorical pairs

In [13]:
inspector.plot_all_associations_heatmap()

Computing associations…



 Heatmap rendered. Hover over cells to see the measure used.


### Step 10 — PlottingMethods Demo
The `PlottingMethods` class provides standalone chart generation that
returns HTML strings for embedding outside of this notebook.

In [14]:
plotter = PlottingMethods()

# Bar chart: survivors by passenger class
survivors_by_class = inspector.df.groupby('Pclass')['Survived'].sum().reset_index()
html1 = plotter.bar_chart(survivors_by_class, x_col='Pclass', y_col='Survived',
                           title='Survivors by Passenger Class')

# Pie chart: embarkation port distribution
port_counts = inspector.df['Embarked'].value_counts().reset_index()
port_counts.columns = ['Port', 'Count']
html2 = plotter.pie_chart(port_counts, names_col='Port', values_col='Count',
                           title='Passengers by Embarkation Port')

# Histogram: age distribution by survival outcome
html3 = plotter.histogram(inspector.df, column='Age',
                           title='Age Distribution by Survival Outcome',
                           nbins=25, color='Survived')

# Save bar chart as a standalone HTML file
plotter.save_html(html1, 'survivors_by_class.html')

 Chart saved to 'survivors_by_class.html'
   Download from Colab: Files panel → right-click → Download


---
##  Demonstration Complete